# M4.16 · Önceden eğitilmiş LM'ler — ELMo, BERT, BART, GPT, RLHF ve DPO

Sabit gömmelerden (word2vec) bağlamsal gömmelere: ELMo'nun çift yönlü LSTM dil modeli ve katman karışımı, GPT-1'in yalnız kod çözücülü ön eğitimi ve tek doğrusal katmanla ince ayarı, BERT'in maskeli dil modeli (15%; 80/10/10) ve sonraki cümle tahmini, BART'ın gürültü giderici kodlayıcı–kod çözücüsü, GPT-2/3'ün ölçek ve sıfır/az örnekli kullanımı, InstructGPT/ChatGPT'nin insan geri bildirimiyle pekiştirmeli öğrenmesi (ödül modeli, KL cezalı ödül) ve DPO. Saf Python'da MLM maskesi ve BERT parametre sayımı, NumPy'da toplu maskeleme, ELMo karışımı, dizi log olasılıkları ve Bradley–Terry ödül modeli, PyTorch'ta sentetik bir dilde MLM ön eğitimi → az etiketle ince ayar, sonda DPO'nun analitik optimum politikaya yakınsaması. Ekler: best-of-n'in KL'si, BART gürültüleri, ELECTRA.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-16-pretrained-lm.html](https://m-gonul.github.io/pytorch-lab/m4-16-pretrained-lm.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_16_pretrained_lm.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t_mlm():\n        rng = random.Random(0)\n        toks = [1] + [5 + (i % 90) for i in range(20000)] + [2]\n        inp, lab = need(ns, \"mlm_mask\")(toks, rng, 100, 3, 4, 0.15)\n        ensure(lab[0] == -100 and lab[-1] == -100 and inp[0] == 1, \"özel token'lar dokunulmaz\")\n        sel = [i for i, l in enumerate(lab) if l != -100]\n        ensure(all(lab[i] == toks[i] for i in sel), \"etiket = orijinal\")\n        f = len(sel) / 20000\n        m = sum(inp[i] == 3 for i in sel) / len(sel)\n        same = sum(inp[i] == toks[i] for i in sel) / len(sel)\n        ensure(abs(f - 0.15) < 0.01 and abs(m - 0.8) < 0.02 and abs(same - 0.1) < 0.02 + 0.01, f\"oranlar seçim {f:.3f}, mask {m:.3f}, aynı {same:.3f}\")\n        ensure(all(inp[i] == toks[i] for i in range(len(toks)) if lab[i] == -100), \"seçilmeyenler değişmez\")\n        P = need(ns, \"nsp_pairs\")([[i] for i in range(400)], random.Random(1))\n        pos = [p for p in P if p[2]]\n        ensure(all(b[0] == a[0] + 1 for a, b, _ in pos) and all(b[0] != a[0] + 1 for a, b, c in P if not c) and abs(len(pos) / len(P) - 0.5) < 0.08, \"NSP çiftleri\")\n    R_.test(\"MLM ve NSP\", t_mlm)\n\n    def t_num():\n        ensure(need(ns, \"bert_params\")(30522, 768, 12, 3072) == 109482240, \"BERT-base\")\n        close(need(ns, \"bt_loss\")(2, 0.5), -math.log(1 / (1 + math.exp(-1.5))), what=\"Bradley–Terry\")\n        close(need(ns, \"kl_reward\")(1, -1.0, -3.0, 0.1), 0.8, what=\"KL cezalı ödül\")\n        close(need(ns, \"dpo_loss\")(-1.0, -5.0, -3.0, -4.0, 0.1), -math.log(1 / (1 + math.exp(-0.3))), what=\"DPO\")\n        rows = []\n        for name, (d, L, ff) in ((\"BERT-base\", (768, 12, 3072)), (\"BERT-large\", (1024, 24, 4096))):\n            p = ns[\"bert_params\"](30522, d, L, ff)\n            rows.append([name, f\"{p / 1e6:.1f} M\", f\"{(30522 + 514) * d / p:.0%}\"])\n        R_.table(\"BERT parametreleri\", [\"model\", \"toplam\", \"embedding payı\"], rows,\n                 note=\"BERT-large makalede 340M; burada 335M (MLM ve NSP başları hariç). Base'de parametrelerin ~%22'si embedding'lerde.\")\n    R_.test(\"sayımlar ve kayıplar\", t_num)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüler\", lambda: ([max_loops(src, 0, func=f) for f in (\"mlm_mask_batch\", \"elmo_mix\", \"seq_logprob\", \"dpo_batch\")], max_loops(src, 1, func=\"fit_reward\")))\n    rng = np.random.default_rng(0)\n\n    def t():\n        X = rng.integers(4, 100, (200, 128))\n        X[:, 0] = 1\n        inp, lab = need(ns, \"mlm_mask_batch\")(X, np.random.default_rng(1), 100, 3, 4)\n        sel = lab != -100\n        ensure(not sel[:, 0].any() and abs(sel.mean() - 0.15 * 127 / 128) < 0.01, \"seçim oranı\")\n        ensure(abs((inp[sel] == 3).mean() - 0.8) < 0.02 and np.all(inp[~sel] == X[~sel]) and np.all(lab[sel] == X[sel]), \"80/10/10\")\n        H = rng.normal(size=(3, 5, 4))\n        close(need(ns, \"elmo_mix\")(H, np.array([0.0, np.log(3), 0.0]), 2.0), 2 * (0.2 * H[0] + 0.6 * H[1] + 0.2 * H[2]), what=\"ELMo karışımı\")\n        logits = rng.normal(size=(2, 4, 6))\n        y = np.array([[1, 2, 3, 4], [5, 0, 1, 2]])\n        m = np.array([[0, 0, 1, 1], [0, 1, 1, 0]])\n        lp = logits - np.log(np.exp(logits).sum(-1, keepdims=True))\n        close(need(ns, \"seq_logprob\")(logits, y, m), [lp[0, 2, 3] + lp[0, 3, 4], lp[1, 1, 0] + lp[1, 2, 1]], what=\"yanıt log olasılığı\")\n        L, acc, mg = need(ns, \"dpo_batch\")(np.array([-1.0, -2.0]), np.array([-5.0, -1.0]), np.array([-3.0, -2.0]), np.array([-4.0, -2.0]), 0.1)\n        close(L, np.mean([np.log1p(np.exp(-0.3)), np.log1p(np.exp(0.1))]), what=\"DPO kaybı\")\n        ensure(acc == 0.5, \"örtük ödül doğruluğu\")\n        wtrue = rng.normal(size=8)\n        Fa, Fb = rng.normal(size=(3000, 8)), rng.normal(size=(3000, 8))\n        pa = 1 / (1 + np.exp(-(Fa - Fb) @ wtrue))\n        win = rng.random(3000) < pa\n        Fw, Fl = np.where(win[:, None], Fa, Fb), np.where(win[:, None], Fb, Fa)\n        w = need(ns, \"fit_reward\")(Fw, Fl, 0.5, 500)\n        cos = w @ wtrue / np.linalg.norm(w) / np.linalg.norm(wtrue)\n        ensure(cos > 0.95, f\"ödül yönü kurtarılır (cos {cos:.3f})\")\n        rows = []\n        for n in (100, 300, 1000, 3000):\n            wn = ns[\"fit_reward\"](Fw[:n], Fl[:n], 0.5, 500)\n            rows.append([n, f\"{wn @ wtrue / np.linalg.norm(wn) / np.linalg.norm(wtrue):.3f}\", f\"{np.linalg.norm(wn) / np.linalg.norm(wtrue):.2f}\"])\n        R_.table(\"Bradley–Terry ödül modeli: yalnız ikili tercihlerden doğrusal ödülü kurtarmak\", [\"tercih çifti\", \"cos(ŵ, w)\", \"‖ŵ‖/‖w‖\"], rows,\n                 note=\"Ödül modeli yalnız farkları görür: r'ye eklenen bir sabit tercihleri değiştirmez (ödül ölçeğe kadar değil, öteleme dışında belirlidir). Az veride ŵ'nin normu şişer (ayrılabilir tercihler); pratikte ödül modelleri erken durdurma ve normalizasyonla eğitilir.\")\n    R_.test(\"maskeleme, ELMo, DPO, ödül modeli\", t)\n",
 "d": "import copy\nimport time\nimport torch\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t_mask():\n        g = torch.Generator().manual_seed(0)\n        x, _ = need(ns, \"corpus\")(2000, 5)\n        inp, lab = need(ns, \"mlm_mask\")(x, g, 64)\n        sel = lab != -100\n        ensure(not sel[:, 0].any() and abs(sel.float().mean().item() - 0.15 * 16 / 17) < 0.01, \"seçim\")\n        ensure(abs((inp[sel] == 3).float().mean().item() - 0.8) < 0.03 and torch.equal(inp[~sel], x[~sel]), \"80/10/10\")\n    R_.test(\"MLM maskesi\", t_mask)\n\n    def t_train():\n        U, _ = ns[\"corpus\"](20000, 0)\n        Xl, yl = ns[\"corpus\"](50, 1)\n        Xt, yt = ns[\"corpus\"](2000, 2)\n        torch.manual_seed(0)\n        pre = need(ns, \"TinyBERT\")(64)\n        t0 = time.perf_counter()\n        L = need(ns, \"pretrain\")(pre, U, 4, 3e-3, 0)\n        tp = time.perf_counter() - t0\n        rows, acc = [], {}\n        for name, mk, fr in ((\"sıfırdan (50 etiket)\", lambda: (torch.manual_seed(1), ns[\"TinyBERT\"](64))[1], False),\n                             (\"MLM ön eğitim + ince ayar\", lambda: copy.deepcopy(pre), False),\n                             (\"MLM ön eğitim, dondurulmuş + doğrusal sonda\", lambda: copy.deepcopy(pre), True)):\n            m = need(ns, \"finetune\")(mk(), Xl, yl, 100, 1e-3, fr)\n            m.eval()\n            with torch.no_grad():\n                acc[name] = (m.cls(m(Xt)[:, 0]).argmax(1) == yt).float().mean().item()\n            rows.append([name, f\"{acc[name]:.3f}\"])\n        a = list(acc.values())\n        ensure(a[1] > a[0] + 0.04 and a[2] > a[0], f\"ön eğitim yardımcı olur {acc}\")\n        R_.table(f\"Konu sınıflandırma, 50 etiketli örnek (ön eğitim: 20 000 etiketsiz cümle, 4 epoch, {tp:.0f} sn, son MLM kaybı {L:.2f})\", [\"yöntem\", \"test doğruluğu\"], rows,\n                 note=\"Etiketsiz metinde maskeli kelimeleri tahmin etmek için model hangi içerik kelimelerinin birlikte geçtiğini (konuyu) ve işlev kelimelerinin içerik kelimelerine bağımlılığını öğrenmek zorunda; bu temsiller az etiketle aşağı akış görevine aktarılır. Dondurulmuş temsillere yalnız doğrusal bir katman eklemek bile sıfırdan eğitmeyi geçer (ELMo/öznitelik tabanlı kullanım); tüm ağırlıkları ince ayarlamak (GPT-1, BERT) daha da iyi.\")\n    R_.test(\"ön eğitim + ince ayar\", t_train)\n",
 "e": "import itertools\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\nV, T = 5, 4\nALL = torch.tensor(list(itertools.product(range(V), repeat=T)))\n\n\ndef reward(y):\n    return (y == 0).sum(-1).float() - (y == 4).sum(-1).float()\n\n\nclass Policy(torch.nn.Module):\n    def __init__(self, seed):\n        super().__init__()\n        g = torch.Generator().manual_seed(seed)\n        self.first = torch.nn.Parameter(torch.randn(V, generator=g) * 0.5)\n        self.trans = torch.nn.Parameter(torch.randn(T - 1, V, V, generator=g) * 0.5)\n\n    def logp(self, y):\n        lp = F.log_softmax(self.first, -1)[y[:, 0]]\n        for t in range(1, T):\n            lp = lp + F.log_softmax(self.trans[t - 1], -1)[y[:, t - 1], y[:, t]]\n        return lp\n\n\ndef check(ns, R_):\n    def t():\n        close(need(ns, \"dpo_loss\")(torch.tensor([-1.0]), torch.tensor([-5.0]), torch.tensor([-3.0]), torch.tensor([-4.0]), 0.1).item(), torch.log1p(torch.exp(torch.tensor(-0.3))).item(), atol=1e-6, what=\"DPO kaybı\")\n        ref = Policy(0)\n        with torch.no_grad():\n            lref = ref.logp(ALL)\n        ensure(abs(lref.exp().sum().item() - 1) < 1e-5, \"625 dizi = tam dağılım\")\n        r = reward(ALL)\n        g = torch.Generator().manual_seed(1)\n        idx = torch.multinomial(lref.exp(), 8000, True, generator=g).view(-1, 2)\n        a, b = ALL[idx[:, 0]], ALL[idx[:, 1]]\n        win = torch.rand(4000, generator=g) < torch.sigmoid(reward(a) - reward(b))\n        yw, yl = torch.where(win[:, None], a, b), torch.where(win[:, None], b, a)\n        with torch.no_grad():\n            rw0, rl0 = ref.logp(yw), ref.logp(yl)\n        rows = []\n        res = {}\n        for beta in (0.25, 0.5, 2.0):\n            pol = Policy(0)\n            opt = torch.optim.Adam(pol.parameters(), 0.05)\n            for _ in range(300):\n                loss = ns[\"dpo_loss\"](pol.logp(yw), pol.logp(yl), rw0, rl0, beta)\n                opt.zero_grad()\n                loss.backward()\n                opt.step()\n            with torch.no_grad():\n                lp = pol.logp(ALL)\n                star = need(ns, \"optimal_logp\")(lref, r, beta)\n                res[beta] = (need(ns, \"kl\")(lp, star).item(), (lp.exp() * r).sum().item(), ns[\"kl\"](lp, lref).item())\n                rows.append([beta, f\"{res[beta][1]:.3f}\", f\"{(star.exp() * r).sum().item():.3f}\", f\"{res[beta][2]:.3f}\", f\"{ns['kl'](star, lref).item():.3f}\", f\"{res[beta][0]:.3f}\"])\n        close(ns[\"kl\"](lref, lref).item(), 0.0, atol=1e-6, what=\"KL(p‖p) = 0\")\n        ensure(res[0.5][0] < 0.1 and res[2.0][0] < 0.05, \"DPO politikası analitik optimuma yakın\")\n        ensure(res[0.25][1] > res[0.5][1] > res[2.0][1] and res[0.25][2] > res[2.0][2], \"β küçüldükçe ödül ve KL artar\")\n        R_.table(f\"DPO, 4000 tercih çifti (referansın beklenen ödülü {(lref.exp() * r).sum().item():.3f})\", [\"β\", \"E_π[r]\", \"E_π*[r]\", \"KL(π‖π_ref)\", \"KL(π*‖π_ref)\", \"KL(π‖π*)\"], rows,\n                 note=\"Gerçek ödül Bradley–Terry ile tercih ürettiğinde DPO'nun optimumu KL cezalı RLHF'inkiyle aynıdır: π* ∝ π_ref · exp(r/β). Burada tüm diziler sayılabildiği için bunu doğrudan doğruluyoruz — ödül modeli, örnekleme ve PPO olmadan. β ödül–KL ödünleşimini ayarlar; küçük β'da sonlu veri ve optimizasyon hatası π'yi π*'dan uzaklaştırır.\")\n    R_.test(\"DPO = kapalı form optimum\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        p = np.array([0.5, 0.3, 0.2])\n        r = np.array([0.0, 2.0, 1.0])\n        q = need(ns, \"bon_dist\")(p, r, 2)\n        close(q, [0.25, 1 - 0.7 ** 2, 0.7 ** 2 - 0.25], what=\"n = 2 örnek\")\n        close(need(ns, \"kl\")(p, p), 0.0, what=\"KL(p‖p)\")\n        rng = np.random.default_rng(0)\n        P = rng.dirichlet(np.ones(5000))\n        R = rng.normal(size=5000)\n        rows = []\n        for n in (1, 2, 4, 16, 64, 256):\n            qn = ns[\"bon_dist\"](P, R, n)\n            k = ns[\"kl\"](qn, P)\n            bound = np.log(n) - (n - 1) / n\n            ensure(k <= bound + 1e-9, \"KL ≤ log n − (n − 1)/n\")\n            rows.append([n, f\"{qn @ R:.3f}\", f\"{k:.3f}\", f\"{bound:.3f}\"])\n        R_.table(\"Best-of-n: 5000 çıktılı referans, ödül ~ N(0, 1)\", [\"n\", \"beklenen ödül\", \"KL(BoN‖ref)\", \"log n − (n−1)/n\"], rows,\n                 note=\"RLHF'e alternatif en basit yöntem: n tamamlama örnekle, ödül modeline göre en iyisini seç. KL n ile yalnız logaritmik büyür, beklenen ödül ise ~√(2 log n) gibi: KL bütçesi başına çok verimlidir ama çıkarımda n kat hesap ister. Sınır sürekli (bağsız) dağılımlarda sıkıdır (Beirami ve ark. 2024).\")\n    R_.test(\"best-of-n\", t)\n",
 "x2": "import random\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = random.Random(0)\n        xs = [need(ns, \"poisson\")(3.0, rng) for _ in range(20000)]\n        m = sum(xs) / len(xs)\n        v = sum((x - m) ** 2 for x in xs) / len(xs)\n        close((m, v), (3.0, 3.0), atol=0.1, what=\"Poisson ortalama = varyans = λ\")\n        ensure(sum(x == 0 for x in xs) / len(xs) > 0.03, \"sıfır uzunluk mümkün\")\n        toks = [f\"w{i}\" for i in range(400)]\n        out = need(ns, \"text_infill\")(toks, 0.3, 3.0, random.Random(1))\n        kept = [t for t in out if t != \"<mask>\"]\n        nm = out.count(\"<mask>\")\n        ensure(kept == [t for t in toks if t in set(kept)], \"kalan token'lar sırayı korur\")\n        dropped = 400 - len(kept)\n        ensure(abs(dropped - 120) <= 3 and nm < dropped, f\"%30 maskelendi ({dropped}), aralık başına tek mask ({nm})\")\n        d = need(ns, \"token_deletion\")(toks, 0.2, random.Random(2))\n        ensure(abs(len(d) - 320) < 30 and all(t in toks for t in d), \"silme\")\n        s = need(ns, \"sentence_permutation\")([[\"a\"], [\"b\"], [\"c\"], [\"d\"]], random.Random(3))\n        ensure(sorted(map(tuple, s)) == [(\"a\",), (\"b\",), (\"c\",), (\"d\",)], \"cümle karıştırma\")\n        r = need(ns, \"document_rotation\")(toks, random.Random(4))\n        k = r.index(\"w0\")\n        ensure(r[k:] + r[:k] == toks, \"döndürme\")\n        R_.note(f\"Örnek: {' '.join(out[:18])} … — {dropped} token {nm} mask ile değiştirildi. BART metin doldurma (span'ler Poisson(λ = 3), %30) + cümle karıştırmayı en iyi bulur: kod çözücü yalnız eksik içeriği değil, eksik uzunluğu da tahmin etmek zorundadır (SpanBERT'ten farkı). Kod çözücü tüm orijinal metni üretir.\")\n    R_.test(\"BART gürültüleri\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        B, T, V = 64, 128, 30\n        X = rng.integers(4, V, (B, T))\n        X[:, 0] = 1\n        good = np.full((B, T, V), 0.1 / (V - 1))\n        np.put_along_axis(good, X[..., None], 0.9, -1)\n        Xc, lab = need(ns, \"electra_corrupt\")(X, np.random.default_rng(1), good, 4)\n        ensure(np.array_equal(lab, Xc != X) and not lab[:, 0].any(), \"etiket = değişti mi\")\n        changed = lab.mean()\n        ensure(0.005 < changed < 0.03, f\"güçlü üreteç: seçilenlerin çoğu orijinale denk ({changed:.3f})\")\n        unif = np.full((B, T, V), 1 / V)\n        Xu, lu = ns[\"electra_corrupt\"](X, np.random.default_rng(2), unif, 4)\n        ensure(0.12 < lu.mean() < 0.16, \"zayıf üreteç: seçilenlerin çoğu değişir\")\n        rows = [[\"MLM (BERT)\", \"%15 (seçilenler)\", \"15%\"], [\"ELECTRA ayırt edici\", \"%100 (her token: orijinal mi?)\", f\"{changed:.1%} / {lu.mean():.1%} değiştirilmiş\"],\n                [\"nedensel LM (GPT)\", \"%100\", \"—\"]]\n        R_.table(\"Token başına öğrenme sinyali\", [\"amaç\", \"kayıp alan konumlar\", \"pozitif (değiştirilmiş) oranı: iyi / düzgün üreteç\"], rows,\n                 note=\"ELECTRA küçük bir MLM üreteciyle maskeli konumları makul token'larla doldurur; ayırt edici her konum için 'değiştirildi mi' sorusunu öğrenir — tüm token'lardan sinyal alır, aynı hesapla BERT'ten belirgin iyidir. Üreteç doğru token'ı örneklediyse o konum 'orijinal' sayılır; üreteç güçlendikçe ayırt edicinin işi zorlaşır (bu yüzden üreteç küçük tutulur).\")\n    R_.test(\"ELECTRA\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m4_16_pretrained_lm as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Bağlamsal gömmeler:** word2vec'in kusuru, bir kelimenin anlamının bağlama bağlı olmasıdır (çok anlamlı kelimeler). Kronolojik sıra (Transformer: NeurIPS 2017):

- **ELMo** (2018): artık bağlantılı çok katmanlı çift yönlü LSTM; ileri LSTM baştan sona her kelimeden sonrakini, geri LSTM sondan başa öncekini tahmin eder (1B Word). Bağlamsal gömme tüm katmanların gizli durumlarının (öğrenilmiş softmax ağırlıklı) birleşimidir: $\gamma\sum_js_jh_j$. Kendi başına görev yapmaz; aşağı akış modeline normal embedding yerine ELMo çıktıları verilir.
- **GPT-1** (2018): kodlayıcısız, kodlayıcı–kod çözücü attention'sız bir Transformer kod çözücüsü; yalnız ileri dil modellemesiyle (BooksCorpus, ~7000 kitap) eğitilir. 12 katman, 768 gizli boyut, 12 baş. Aşağı akış görevine tek bir tam bağlı katman eklenip **tüm** ağırlıklar ince ayarlanır; girdiler token dizilerine çevrilir.
- **BERT** (2019): çift yönlü (ELMo gibi) ve Transformer (GPT gibi); base 110M (12/768/12), large 340M (24/1024/16). İki görev: (1) **maskeli token tahmini** — komşulardan maskelenmiş kelimeyi tahmin et, (2) **sonraki cümle tahmini** — iki cümle ardışık mı? Görevler için ince ayar gerekir.
- **BART** (2019): kodlayıcı–kod çözücülü gürültü giderici otomatik kodlayıcı. BERT = kodlayıcı, GPT = kod çözücü, BART = ikisi.
- **GPT-2** (2019): 1.5B (48 katman, 1600, bağlam 1024), WebText (Reddit'te ≥ 3 karma alan bağlantılar, 8M belge, 40 GB); ince ayarsız, yalnız istemle (prompt) çok görev. **GPT-3** (2020): 175B (96/12 288/96 baş, bağlam 2048), ≥ 570 GB; sıfır/bir/az örnekli öğrenme.
- **InstructGPT** (2022): GPT-3 kullanıcı niyetini yakalayamıyordu — **RLHF**: (1) gösterimlerle denetimli ince ayar, (2) insan karşılaştırmalarından **ödül modeli** (Bradley–Terry: $P(y_w\succ y_l)=\sigma(r(y_w)-r(y_l))$), (3) KL cezalı ödülle ($r-\beta\log\pi/\pi_{ref}$) PPO. **ChatGPT** (30 Kasım 2022): GPT-3.5 + RLHF, sohbet. **DPO** (2023): KL cezalı amacın optimumu $\pi^*(y)\propto\pi_{ref}(y)e^{r(y)/\beta}$; ödülü $r=\beta\log\frac{\pi}{\pi_{ref}}+\text{sabit}$ olarak yazıp Bradley–Terry'ye koyunca ödül modeli ve RL olmadan:
$$\mathcal L_{DPO}=-\log\sigma\Big(\beta\log\tfrac{\pi_\theta(y_w)}{\pi_{ref}(y_w)}-\beta\log\tfrac{\pi_\theta(y_l)}{\pi_{ref}(y_l)}\Big).$$

## 1 · Elle

<!-- çözüm -->

MLM: 512 · 0.15 = 76.8 seçilir; %80'i (61.4) [MASK], %10'u (7.68) rastgele token, %10'u değişmeden kalır. Kayıp yalnız seçilen 76.8 konumda hesaplanır — token başına sinyal nedensel LM'in ~1/7'si (ELECTRA'nın motivasyonu, x3).

BERT-base: embedding $(30522+512+2)\cdot768+2\cdot768=23\,837\,184$; katman $7\,087\,872$ (ViT ile aynı blok) × 12 = 85 054 464; pooler $768^2+768=590\,592$ → **109 482 240** ≈ 110M.

GPT-2: $12\cdot48\cdot1600^2=1.47\cdot10^9$ (+ embedding 50 257 · 1600 ≈ 80M → 1.5B). GPT-3: $12\cdot96\cdot12288^2=1.74\cdot10^{11}$ ≈ 175B.

Ödül modeli: $-\log\sigma(1.5)=0.201$. KL cezalı: $1-0.1\cdot2=0.8$. DPO: $-\log\sigma(0.1\cdot(2-(-1)))=-\log\sigma(0.3)=0.554$; gradyan ağırlığı $\sigma(-0.3)=0.43$ — örtük ödül sırayı ne kadar yanlış tahmin ediyorsa o kadar büyük.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**BERT**

- `sel`: 512 token'lık dizide MLM için seçilen token sayısının beklentisi
- `rnd`: Bunlardan rastgele token'la değiştirilenlerin beklentisi
- `bert`: BERT-base parametresi: V = 30 522, 512 konum, 2 segment, d = 768, 12 katman, FFN 3072, pooler dahil (MLM başı hariç)

**GPT ölçeği ($\approx12Ld^2$)**

- `gpt2`: GPT-2: L = 48, d = 1600
- `gpt3`: GPT-3: L = 96, d = 12 288

**RLHF ve DPO**

- `bt`: Ödül modeli: $r(y_w)=2$, $r(y_l)=0.5$ → $-\log\sigma(r_w-r_l)$
- `kl`: KL cezalı ödül $r-\beta\log(\pi/\pi_{ref})$: r = 1, β = 0.1, log oran 2
- `dpo`: DPO, β = 0.1: $\log\frac{\pi_\theta}{\pi_{ref}}(y_w)=2$, $(y_l)=-1$ → kayıp

In [ ]:
cevaplar = dict(
    sel=None,
    rnd=None,
    bert=None,
    gpt2=None,
    gpt3=None,
    bt=None,
    kl=None,
    dpo=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"sel":[76.8,1e-9],"rnd":[7.68,1e-9],"bert":[109482240,0],"gpt2":[1474560000,0],"gpt3":[173946175488,0],"bt":[0.20141327798275246,0.000001],"kl":[0.8,1e-9],"dpo":[0.5543552444685271,0.000001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `mlm_mask(tokens, rng, V, mask_id, n_special, p=0.15)` → `(girdi, etiket)`: id'si `< n_special` olan özel token'lar hiç seçilmez; diğerleri p olasılıkla seçilir; seçilenlerin %80'i `mask_id`, %10'u `rng.randrange(n_special, V)`, %10'u aynı kalır. Etiket seçilenlerde orijinal token, diğerlerinde −100. `rng`: `random.Random`.
- `nsp_pairs(sentences, rng)` → `[(A, B, ardışık_mı)]`: her i < n − 1 için %50 `(s_i, s_{i+1}, True)`, %50 `(s_i, rastgele s_j (j ≠ i + 1), False)`.
- `bert_params(V, d, L, ff, max_pos=512, types=2)` — embedding'ler + embedding LN + L katman + pooler.
- `bt_loss(rw, rl)`, `kl_reward(r, logp, logp_ref, beta)`, `dpo_loss(lp_w, lp_l, ref_w, ref_l, beta)`.

In [ ]:
%%writefile stage_b.py
import math
import random


def mlm_mask(tokens, rng, V, mask_id, n_special, p=0.15):
    ...


def nsp_pairs(sentences, rng):
    ...


def bert_params(V, d, L, ff, max_pos=512, types=2):
    ...


def bt_loss(rw, rl):
    ...


def kl_reward(r, logp, logp_ref, beta):
    ...


def dpo_loss(lp_w, lp_l, ref_w, ref_l, beta):
    ...

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy.

- `mlm_mask_batch(X, rng, V, mask_id, n_special, p=0.15)` — döngüsüz; `rng`: `np.random.Generator`.
- `elmo_mix(H, s, gamma)` — `H (L+1, T, d)`, `s (L+1,)` ham ağırlıklar → $\gamma\sum_j\mathrm{softmax}(s)_jH_j$.
- `seq_logprob(logits, y, resp_mask)` — `logits (B, T, V)` (t. konum y_t'yi tahmin eder), yanıt token'larının log olasılık toplamı `(B,)`.
- `dpo_batch(pw, pl, rw, rl, beta)` → `(ortalama kayıp, örtük ödül doğruluğu, ortalama marj)`; örtük ödül $\beta(\log\pi-\log\pi_{ref})$.
- `fit_reward(Fw, Fl, lr=0.5, iters=500)` — doğrusal ödül $r=Fw$; Bradley–Terry kaybında gradyan inişi (tek döngü).

In [ ]:
%%writefile stage_c.py
import numpy as np


def mlm_mask_batch(X, rng, V, mask_id, n_special, p=0.15):
    ...


def elmo_mix(H, s, gamma):
    ...


def seq_logprob(logits, y, resp_mask):
    ...


def dpo_batch(pw, pl, rw, rl, beta):
    ...


def fit_reward(Fw, Fl, lr=0.5, iters=500):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch. Sentetik dil: `corpus(n, seed)` verilir — `[CLS]` + 16 token; çift konumlarda "işlev" kelimeleri (bir sonraki içerik kelimesinin belirlenimci fonksiyonu), tek konumlarda içerik kelimeleri (cümlenin konusuna ait 8 kelimeden ya da %50 olasılıkla tüm konulardan rastgele). Aşağı akış görevi: konu (5 sınıf).

- `mlm_mask(x, g, V, p=0.15)` — torch sürümü (özel token'lar 0–3: PAD, CLS, SEP, MASK).
- `TinyBERT(V, d=64, L=2)` — embedding + öğrenilebilir konum + Pre-LN kodlayıcı + LN; `mlm` başı (d → V) ve `cls` başı (d → 5); `forward(x)` gizli durumlar.
- `pretrain(model, U, epochs, lr, seed)` — MLM kaybı (yalnız seçilen konumlar).
- `finetune(model, X, y, steps, lr, freeze=False)` — `[CLS]` durumu → `cls` başı; `freeze=True`: yalnız baş (doğrusal sonda).

In [ ]:
%%writefile stage_d.py
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

PAD, CLS, SEP, MASK = 0, 1, 2, 3
V = 64


def corpus(n, seed, T=16, K=5):
    g = torch.Generator().manual_seed(seed)
    topic = torch.randint(0, K, (n,), generator=g)
    content = 24 + topic[:, None] * 8 + torch.randint(0, 8, (n, T // 2), generator=g)
    noise = torch.rand(n, T // 2, generator=g) < 0.5
    content = torch.where(noise, 24 + torch.randint(0, K * 8, (n, T // 2), generator=g), content)
    func = 4 + (content * 7 + torch.arange(T // 2)) % 20
    seq = torch.stack([func, content], 2).flatten(1)
    return torch.cat([torch.full((n, 1), CLS), seq], 1), topic


def mlm_mask(x, g, V, p=0.15):
    ...


class TinyBERT(nn.Module):
    def __init__(self, V, d=64, L=2):
        super().__init__()
        ...

    def forward(self, x):
        ...


def pretrain(model, U, epochs, lr, seed):
    ...


def finetune(model, X, y, steps, lr, freeze=False):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

DPO'yu tam izlenebilir bir oyuncakta çalıştırmak: sözlük 5, uzunluk 4 → tüm 625 dizi sayılabilir. Politika: ilk token için logit'ler + konum başına geçiş matrisleri (verilir). Gerçek ödül: 0 sayısı − 4 sayısı; tercihler referanstan örneklenen çiftlerde Bradley–Terry ile.

- `dpo_loss(pw, pl, rw, rl, beta)` — torch, ortalama.
- `optimal_logp(lref, r, beta)` — $\log\pi^*=\log\mathrm{softmax}(\log\pi_{ref}+r/\beta)$ (tüm diziler üzerinde).
- `kl(lp, lq)` — $\sum e^{lp}(lp-lq)$.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def dpo_loss(pw, pl, rw, rl, beta):
    ...


def optimal_logp(lref, r, beta):
    ...


def kl(lp, lq):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Best-of-n örnekleme ve KL bütçesi

NumPy. Ayrık çıktılar: referans olasılıkları `p`, birbirinden farklı ödüller `r`.

- `bon_dist(p, r, n)` — n örnekten en yüksek ödüllüyü seçmenin dağılımı: ödüle göre sıralı kümülatif F ile $F(y)^n-F(y^-)^n$.
- `kl(q, p)`.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def bon_dist(p, r, n):
    ...


def kl(q, p):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · BART'ın gürültü fonksiyonları

Saf Python, `random.Random`.

- `poisson(lam, rng)` — Knuth.
- `text_infill(tokens, ratio, lam, rng, mask="<mask>")` — maskelenmiş token oranı ≈ `ratio` olana kadar: uzunluğu Poisson(λ) olan bir aralık seç (rastgele başlangıç, henüz maskelenmemiş), tamamını **tek** bir mask ile değiştir; 0 uzunluk = araya mask ekle. Sonuç listesi.
- `token_deletion(tokens, p, rng)`, `sentence_permutation(sentences, rng)`, `document_rotation(tokens, rng)`.

In [ ]:
%%writefile extra_x2.py
import math
import random


def poisson(lam, rng):
    ...


def text_infill(tokens, ratio, lam, rng, mask="<mask>"):
    ...


def token_deletion(tokens, p, rng):
    ...


def sentence_permutation(sentences, rng):
    ...


def document_rotation(tokens, rng):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · ELECTRA: değiştirilmiş token tespiti

NumPy.

- `electra_corrupt(X, rng, gen_probs, n_special, p=0.15)` — konumların p'sini seç (özel token'lar hariç), seçilenlere üretecin dağılımından (`gen_probs (B, T, V)`) örnek koy → `(bozulmuş, etiket)`; etiket = bozulmuş ≠ orijinal (üreteç doğru token'ı örneklediyse "orijinal" sayılır).

In [ ]:
%%writefile extra_x3.py
import numpy as np


def electra_corrupt(X, rng, gen_probs, n_special, p=0.15):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Devlin ve ark. (2019): küçük bir BERT

[Devlin, Chang, Lee, Toutanova — BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding (NAACL 2019)](https://arxiv.org/abs/1810.04805)

d aşamasındaki kodlayıcıyı WikiText-103'ün bir alt kümesinde (BPE, M4.5) MLM + NSP ile ön eğit (L = 4, d = 256); SST-2 ve MRPC'de (GLUE) ince ayar yap. Makale Tablo 5'in küçük ablasyonu: (a) NSP'siz, (b) yalnız soldan sağa LM (GPT tarzı, nedensel maske), (c) 80/10/10 yerine %100 [MASK] (Tablo 8). Ayrıca öznitelik tabanlı kullanımı (dondurulmuş katmanların ağırlıklı toplamı, ELMo tarzı) ince ayarla karşılaştır.


**Adımlar**

1. Ön eğitim adım sayısını sabit tut; ablasyonları aynı bütçeyle eğit.

**Kabul ölçütleri**

- Çift yönlü MLM, soldan sağa LM'den GLUE'da iyi.
- %100 [MASK] özellikle öznitelik tabanlı kullanımda geride kalır.

## Makale ödevi: Rafailov ve ark. (2023): DPO'yu küçük bir LM'de

[Rafailov, Sharma, Mitchell, Ermon, Manning, Finn — Direct Preference Optimization: Your Language Model is Secretly a Reward Model (NeurIPS 2023)](https://arxiv.org/abs/2305.18290)

Makale Bölüm 6.1'in "kontrollü duygu üretimi" deneyini küçük ölçekte tekrarla: GPT-2 small'ı (ya da M8.1'deki mini-LLM'i) IMDb film yorumu önekleriyle referans politika olarak kullan; bir duygu sınıflandırıcısını "gerçek ödül" say, tercih çiftlerini ondan üret. DPO ile PPO-RLHF'i (ödül modeli + KL cezalı PPO, M11.12) ödül–KL sınırında (Şekil 2 sol) karşılaştır; β ve tercih verisi miktarını değiştir.


**Adımlar**

1. KL'yi tamamlama düzeyinde $\sum_t\log\pi_\theta/\pi_{ref}$ ile tahmin et.

**Kabul ölçütleri**

- DPO aynı KL'de PPO'dan yüksek ya da eşit ödül verir.
- β küçüldükçe ödül ve KL birlikte artar (sınır eğrisi).

## Kaynaklar

- [Peters ve ark. — Deep contextualized word representations (ELMo, NAACL 2018)](https://arxiv.org/abs/1802.05365) — arXiv
- [Radford ve ark. — Improving Language Understanding by Generative Pre-Training (GPT-1, 2018)](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) — ©
- [Devlin ve ark. — BERT (NAACL 2019)](https://arxiv.org/abs/1810.04805) — arXiv
- [Lewis ve ark. — BART (ACL 2020)](https://arxiv.org/abs/1910.13461) — arXiv
- [Brown ve ark. — Language Models are Few-Shot Learners (GPT-3, NeurIPS 2020)](https://arxiv.org/abs/2005.14165) — arXiv
- [Ouyang ve ark. — Training language models to follow instructions with human feedback (InstructGPT, 2022)](https://arxiv.org/abs/2203.02155) — arXiv
- [Rafailov ve ark. — Direct Preference Optimization (NeurIPS 2023)](https://arxiv.org/abs/2305.18290) — arXiv
- [Clark ve ark. — ELECTRA (ICLR 2020)](https://arxiv.org/abs/2003.10555) — arXiv
- [Hugging Face LLM Course — Transformer models, fine-tuning](https://huggingface.co/learn/llm-course) — Apache-2.0
- [Stanford CS224n — Pretraining; Post-training (RLHF, DPO)](https://web.stanford.edu/class/cs224n/) — ©